# ADAptive GRADient (ADAGRAD)

In [1]:
import nnfs
from nnfs.datasets import spiral_data
import numpy as np
import matplotlib.pyplot as plt
nnfs.init()

In [2]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01*np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1,n_neurons))
    def forward(self, X):
        self.inputs = X
        self.outputs = np.dot(X, self.weights) + self.biases
    def backward(self, dvalues):
        #dvalues is the gradient of the loss wrt the output of the layer
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)
        
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.outputs = np.maximum(0, inputs)
    def backward(self, dvalues):
        #First copy and then check <0 or >0
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0 
        
class Activation_Softmax:
    def forward(self, X):
        not_normalized = np.exp(X - np.max(X, axis = 1, keepdims=True))
        self.outputs = not_normalized/np.sum(not_normalized, axis = 1, keepdims=True)
        
class Loss:
    def calculate(self, y_pred, y_true):
        return np.mean(self.forward(y_pred, y_true))
    
class CategoricalCrossEntropy(Loss): #CCE inherits Loss
    def forward(self, y_pred, y_true):
        #Clip y_pred
        y_pred_clip = np.clip(y_pred, 1e-7, 1 - 1e-7)
        if len(y_true.shape) == 1: #Case 1: Sparse format [0,1,1,2]
            return -np.log(y_pred_clip[range(len(y_pred_clip)), y_true])
        elif len(y_true.shape) == 2: #Case 2: One Hot Encoding of Output
            return -np.log(np.sum(y_pred_clip*y_true, axis = 1)) #Element-wise multiplication
        #Note you are returning negative log likelihood matrix
    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])
    
        #If class labels are sparse, convert to one hot encoded
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]
        
        self.dinputs = - y_true/dvalues         #Gradient Calculation
        self.dinputs = self.dinputs/samples     #Normalized with number of samples
        
class Activation_Softmax_CategoricalCrossEntropy:
    def __init__(self):
        self.loss = CategoricalCrossEntropy()
        self.activation = Activation_Softmax()
        
    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.outputs = self.activation.outputs
        return self.loss.calculate(self.outputs, y_true)
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        
        #Convert one-hot encoded y_true to sparse form
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)
        
        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        
        #Normalize this
        self.dinputs = self.dinputs/samples
    
class ADAGRAD:
    def __init__(self, learning_rate = 1.0, decay = 0, epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate 
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
    
    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = (self.learning_rate) / (1. + self.decay*self.iterations)
    
    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)
        
        layer.weight_cache += layer.dweights**2
        layer.bias_cache += layer.dbiases**2
        
        layer.weights += -((self.current_learning_rate*layer.dweights)/np.sqrt(layer.weight_cache + self.epsilon))
        layer.biases += -((self.current_learning_rate*layer.dbiases)/np.sqrt(layer.bias_cache + self.epsilon))
        
    def post_update_params(self):
        self.iterations += 1

In [8]:
X, y = spiral_data(samples = 100, classes = 3)

dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_CategoricalCrossEntropy()
optimizer = ADAGRAD(decay = 1e-4)

for epoch in range(10000):
    dense1.forward(X)
    activation1.forward(dense1.outputs)
    dense2.forward(activation1.outputs)
    loss = loss_activation.forward(dense2.outputs, y)
    y_pred = np.argmax(loss_activation.outputs, axis = 1)
    
    if len(y.shape) == 2:
        y_true = np.argmax(y, axis = 1)
    else:
        y_true = y
    
    accuracy = np.mean(y_true == y_pred)
    
    if (epoch+1)% 100 == 0:
        print(f"Epoch: {epoch+1}\tAccuracy: {accuracy:.3f}\tLoss:{loss:.3f}\tLearning Rate:{optimizer.current_learning_rate:.5f}")
    
    loss_activation.backward(loss_activation.outputs,y_true)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

Epoch: 100	Accuracy: 0.470	Loss:0.961	Learning Rate:0.99030
Epoch: 200	Accuracy: 0.490	Loss:0.923	Learning Rate:0.98058
Epoch: 300	Accuracy: 0.527	Loss:0.885	Learning Rate:0.97106
Epoch: 400	Accuracy: 0.570	Loss:0.847	Learning Rate:0.96172
Epoch: 500	Accuracy: 0.603	Loss:0.813	Learning Rate:0.95256
Epoch: 600	Accuracy: 0.643	Loss:0.760	Learning Rate:0.94357
Epoch: 700	Accuracy: 0.690	Loss:0.733	Learning Rate:0.93475
Epoch: 800	Accuracy: 0.700	Loss:0.697	Learning Rate:0.92610
Epoch: 900	Accuracy: 0.563	Loss:0.775	Learning Rate:0.91760
Epoch: 1000	Accuracy: 0.667	Loss:0.638	Learning Rate:0.90926
Epoch: 1100	Accuracy: 0.713	Loss:0.604	Learning Rate:0.90106
Epoch: 1200	Accuracy: 0.703	Loss:0.595	Learning Rate:0.89302
Epoch: 1300	Accuracy: 0.703	Loss:0.590	Learning Rate:0.88511
Epoch: 1400	Accuracy: 0.743	Loss:0.549	Learning Rate:0.87735
Epoch: 1500	Accuracy: 0.737	Loss:0.549	Learning Rate:0.86972
Epoch: 1600	Accuracy: 0.737	Loss:0.541	Learning Rate:0.86222
Epoch: 1700	Accuracy: 0.740	Loss: